# Mini-Project 1: UBOS District Population Forecaster
**Scenario.** A district planning unit wants 5-year population forecasts (2025–2029) to plan primary-school classrooms. Ten years of estimates (2015–2024, thousands) are given for Kampala, Wakiso and Gulu. **The Mbarara and Jinja series are invented for this exercise** (illustrative, not UBOS figures); they were chosen so that one grows steadily and one is slowing down.

Logic lives in `src/population.py` (`DistrictPopulation`, abstract `Forecaster` with three subclasses, `ModelSelector`, `ClassroomPlanner`); tests are in `tests/test_population.py`.

In [ ]:
# --- Google Colab setup: does nothing when you run on your own computer ---
import os, subprocess, sys

REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"   # change this after you push the repo to GitHub

if "google.colab" in sys.modules and not os.path.isdir("src"):
    if "YOUR_USERNAME" in REPO_URL:
        raise RuntimeError("Colab cannot find the repo's src/ folder. Either edit REPO_URL above, or upload and "
                           "unzip the repo first (see the README section 'Run in Google Colab').")
    repo_dir = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.isdir(repo_dir):
        subprocess.run(["git", "clone", "-q", REPO_URL, repo_dir], check=True)
    os.chdir(repo_dir)
print("Working folder:", os.getcwd())

In [ ]:
import statistics
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from src.population import (
    CAGRForecaster, ClassroomPlanner, DistrictForecast, DistrictPopulation, FibonacciRatioForecaster,
    Forecaster, LinearTrendForecaster, ModelSelector, plot_district_forecasts,
)

SEED = 42
N_BOOT = 2_000                      # bootstrap resamples (assignment asks for >= 1,000)
Path("figures").mkdir(exist_ok=True)
YEARS = list(range(2015, 2025))
RAW = {
    "Kampala": [1200, 1250, 1300, 1350, 1420, 1500, 1580, 1650, 1720, 1800],
    "Wakiso":  [ 950, 1000, 1070, 1150, 1220, 1300, 1390, 1480, 1570, 1670],
    "Gulu":    [ 320,  330,  345,  360,  375,  390,  410,  430,  455,  480],
    "Mbarara": [ 470,  485,  500,  520,  540,  560,  585,  610,  635,  660],   # invented
    "Jinja":   [ 440,  458,  474,  488,  500,  510,  518,  525,  530,  534],   # invented (slowing growth)
}
districts = {name: DistrictPopulation(name, YEARS, pop) for name, pop in RAW.items()}
list(districts.values())

## Task 1: `DistrictPopulation`: validation, `__repr__`, `__len__`

In [ ]:
d = districts["Kampala"]
print(repr(d), "| len =", len(d), "| arrays:", type(d.years).__name__, type(d.population).__name__)
bad_inputs = {
    "unequal lengths": (YEARS, RAW["Kampala"][:-1]),
    "negative value":  (YEARS, [-1] + RAW["Kampala"][1:]),
    "NaN value":       (YEARS, [float("nan")] + RAW["Kampala"][1:]),
    "years not increasing": ([2015, 2015] + YEARS[2:], RAW["Kampala"]),
}
for label, (yrs, pop) in bad_inputs.items():
    try:
        DistrictPopulation("Bad", yrs, pop)
    except ValueError as err:
        print(f"{label:<22} -> ValueError: {err}")

## Task 2: Mean, median, variance and st.dev: `statistics` module versus NumPy

In [ ]:
print(f"{'district':<9}{'mean':>8}{'median':>8} | {'statistics.variance':>20}{'np.var ddof=0':>15}{'np.var ddof=1':>15} | {'statistics.stdev':>17}{'np.std ddof=0':>15}")
for name, d in districts.items():
    s, n0, n1 = d.stats_statistics(), d.stats_numpy(ddof=0), d.stats_numpy(ddof=1)
    assert np.isclose(s["variance"], n1["variance"]) and np.isclose(s["pvariance"], n0["variance"])
    print(f"{name:<9}{s['mean']:8.1f}{s['median']:8.1f} | {s['variance']:20,.1f}{n0['variance']:15,.1f}{n1['variance']:15,.1f} | {s['stdev']:17.2f}{n0['stdev']:15.2f}")
k = districts["Kampala"].stats_statistics()
print(f"\nKampala: population variance x n/(n-1) = {k['pvariance']:,.1f} x 10/9 = {k['pvariance']*10/9:,.1f} = statistics.variance ({k['variance']:,.1f})")

**Why the numbers differ, and what `ddof` does.** The two functions use different divisors.
- `statistics.variance` (and `stdev`) is the **sample** variance: Σ(x − x̄)² ÷ **(n − 1)**. It treats the ten values as a sample from a larger process and corrects for the fact that the mean x̄ was estimated from the same data (which uses up one degree of freedom; Bessel's correction).
- `np.var` defaults to the **population** variance: Σ(x − x̄)² ÷ **n** (`ddof = 0`). `statistics.pvariance` is its twin.
- `ddof` ("delta degrees of freedom") sets the divisor to **n − ddof**. With `ddof = 1`, NumPy reproduces `statistics.variance` exactly (the assertion above), and the two variances always differ by the factor n/(n − 1) = 10/9 here.
- Mean and median agree because they do not depend on a divisor choice. A caution for later: these are *trending* series, so their variance mostly measures how far the trend has travelled, not volatility (see Task 6).

## Task 3: Year-on-year growth and CAGR

In [ ]:
print(f"{'':<9}" + "".join(f"{y:>7}" for y in YEARS[1:]) + f"{'CAGR':>9}")
for name, d in districts.items():
    print(f"{name:<9}" + "".join(f"{g*100:6.2f}%" for g in d.yoy_growth()) + f"{d.cagr()*100:8.2f}%")
ranked = sorted(districts.values(), key=lambda x: x.cagr(), reverse=True)
print("\nRanking by CAGR:", ", ".join(f"{x.name} {x.cagr():.2%}" for x in ranked))
print(f"Jinja: growth fell from {districts['Jinja'].yoy_growth()[0]:.2%} to {districts['Jinja'].yoy_growth()[-1]:.2%} a year")

CAGR = (P₂₀₂₄ / P₂₀₁₅)^(1/9) − 1 (nine growth periods between ten annual observations). **Wakiso is growing fastest in relative terms (6.47% a year)**, even though Kampala is bigger in absolute numbers. Kampala and Gulu have the *same* CAGR (4.61%) because both grew by exactly 50% over the decade. CAGR uses only the two end points, so it hides the path: Jinja's CAGR of 2.17% conceals a slowdown to under 1% a year.

## Task 4: Three forecasters behind an abstract `Forecaster`
`Forecaster` is an abstract base class: `fit(years, values)` and `predict(horizon)` are shared and validate their input once; each subclass supplies only its model (`_fit`, `_predict_years`, `_fitted`).
- **`LinearTrendForecaster`**: line fitted with `np.polyfit(years, values, 1)`.
- **`CAGRForecaster`**: exponential growth at the training-period CAGR, anchored on the last observation: P(last + h) = P(last)·(1 + CAGR)^h.
- **`FibonacciRatioForecaster`**: scales the **last value** by successive Fibonacci ratios F(k+h)/F(k+h−1), starting at the beginning of the sequence: 1/1, 2/1, 3/2, 5/3, 8/5, … That is `forecast_h = last × ratio_h`. *Assumption:* the assignment describes this model only briefly; I implement the literal reading and also test a compounding reading (`chain=True`) in the extension.

In [ ]:
for cls in (LinearTrendForecaster, CAGRForecaster, FibonacciRatioForecaster):
    print(f"{cls.__name__:<26} subclass of Forecaster: {issubclass(cls, Forecaster)}")
try:
    Forecaster()
except TypeError as err:
    print("\nForecaster() ->", err)

models = [LinearTrendForecaster(), CAGRForecaster(), FibonacciRatioForecaster()]
d = districts["Kampala"]
print("\nKampala, forecast of 2025-2027 from all ten years:")
for m in models:
    print(f"  {m.name:<20}", m.clone().fit_district(d).predict(3).round(0))

## Task 5: Validate before you forecast: train 2015–2021, test 2022–2024
Each model is fitted on 2015–2021 only and scored on the held-out 2022–2024. **Selection criterion: RMSE** (same units as the data and punishes large misses; MAE and MAPE are reported alongside and I check whether they agree).

In [ ]:
selector = ModelSelector(models, last_train_year=2021, criterion="rmse")
winners = {}
for name, d in districts.items():
    scores = selector.evaluate(d)
    best = selector.best(scores)
    winners[name] = best.model
    agree = all(ModelSelector(models, 2021, c).best(scores).model == best.model for c in ("mae", "mape"))
    print(f"\n{name}   (test actuals 2022-24: {d.population[-3:].astype(int).tolist()})")
    print(f"  {'model':<20}{'MAE':>10}{'RMSE':>10}{'MAPE %':>9}   forecast 2022-24")
    for s in scores:
        flag = "  <- best" if s.model == best.model else ""
        print(f"  {s.model:<20}{s.mae:10.2f}{s.rmse:10.2f}{s.mape:9.2f}   {s.forecast.round(0).tolist()}{flag}")
    print(f"  MAE and MAPE pick the same model: {agree}")
print("\nSelected:", winners)

**Reading the tables.**
- The Fibonacci-ratio model is wrong by **36–47% (MAPE)** in every district, roughly ten times or more worse than the other two models (0.4–4%). Its forecasts are not even monotone (e.g. Kampala: 1,580 → 3,160 → 2,370).
- **CAGR beats the linear trend in all five districts** (and all three criteria agree). The series are growing at a roughly steady *percentage* rate, so they curve upward, and a straight line under-predicts them.
- **Jinja is nearly a tie** (RMSE 19.4 vs 20.5): its growth is slowing, so both models overshoot. With only three test points, small gaps like this are not strong evidence.

## Task 6: Forecast 2025–2029 with the selected model
After validation, each selected model is **refitted on all ten years** (2015–2024) before forecasting. Then the variance of the actual series is compared with the variance of the forecast series.

In [ ]:
HORIZON = 5
final = {}
print(f"{'district':<9}{'model':<20}" + "".join(f"{y:>9}" for y in range(2025, 2030)))
for name, d in districts.items():
    model = selector.select(d).fit_district(d)
    final[name] = model
    print(f"{name:<9}{model.name:<20}" + "".join(f"{v:9,.0f}" for v in model.predict(HORIZON)))

print()
print(f"{'district':<9}{'var actual 15-24':>17}{'var actual 20-24':>17}{'var forecast 25-29':>19}{'fc / last-5':>12}{'resid var (actual)':>20}{'resid var (forecast)':>21}")
for name, d in districts.items():
    m = final[name]; f = m.predict(HORIZON)
    v_act, v_last5, v_fc = (statistics.variance(x) for x in (d.population, d.population[-5:], f))
    resid_var = statistics.variance(d.population - m.fitted_values())
    print(f"{name:<9}{v_act:17,.1f}{v_last5:17,.1f}{v_fc:19,.1f}{v_fc/v_last5:12.2f}{resid_var:20.1f}{0.0:21.1f}")

**What the difference tells us.**
1. **Variance here measures the trend, not volatility.** For points on a straight line with slope *b*, the sample variance is b²·n(n+1)/12, so it grows with both the slope and the number of points. The actual series has 10 points, the forecast only 5, so the forecast's variance is smaller than the 10-year variance in every district (37–67% of it). That says nothing about the future being calmer.
2. **Like for like (last 5 actual years vs the 5 forecast years), the forecast variance is *higher* in every district (1.2–4.1×)** because exponential growth accelerates: each year adds more people than the last, so the forecast covers a steeper slope.
3. **The forecast contains no noise.** The actual series wiggles around its fitted curve (residual variance 9–99 in thousands²); the forecast sits exactly on the model curve (residual variance 0 by construction). A forecast is an expected path, so its variance **understates** real uncertainty. The right way to show uncertainty is a prediction interval (extension below), not the variance of the forecast series.

## Task 7: Actual, fitted and forecast on one figure (train/test split marked)

In [ ]:
results = []
intervals = {}
for name, d in districts.items():
    train, test = d.split(2021)
    winner = final[name]
    train_model = winner.clone().fit_district(train)
    iv = winner.bootstrap_interval(HORIZON, n_boot=N_BOOT, level=0.95, seed=SEED)      # extension
    intervals[name] = iv
    results.append(DistrictForecast(
        district=d, model_name=winner.name, split_year=2021,
        train_fitted=train_model.fitted_values(), test_forecast=train_model.predict(len(test)),
        future_years=winner.forecast_years(HORIZON), future_values=winner.predict(HORIZON),
        lower=iv["lower"], upper=iv["upper"]))
fig = plot_district_forecasts(results, ncols=3)
fig.savefig("figures/project1_forecasts.png", dpi=150)
plt.show()

In each panel the shaded band is the held-out test period (2022–24), the red squares are what the training-only model predicted for it, and the green triangles with the shaded band are the final 2025–29 forecast with its 95% bootstrap interval. The red squares sit close to the black actuals for Kampala, Wakiso and Mbarara, are slightly low for Gulu, and visibly overshoot for Jinja, whose growth is slowing.

## Task 8: Classrooms needed by 2029
**Assumptions.** 18% of the population is primary-school age; a classroom holds 53 pupils; population is in thousands (×1,000); classrooms are rounded **up**; today's classroom stock is assumed to exactly meet the 2024 need (no existing surplus or backlog); the 18% share and enrolment rate stay constant.

In [ ]:
planner = ClassroomPlanner(primary_share=0.18, pupils_per_classroom=53)
print(f"{'district':<9}{'pop 2024 (k)':>13}{'pop 2029 (k)':>13}{'classrooms 2024':>16}{'classrooms 2029':>16}{'additional':>11}{'95% range':>14}")
additional = {}
for name, d in districts.items():
    f29, iv = final[name].predict(HORIZON)[-1], intervals[name]
    now, then = planner.classrooms_needed(d.population[-1]), planner.classrooms_needed(f29)
    lo, hi = (planner.additional_classrooms(d.population[-1], iv[k][-1]) for k in ("lower", "upper"))
    additional[name] = planner.additional_classrooms(d.population[-1], f29)
    print(f"{name:<9}{d.population[-1]:13,.0f}{f29:13,.0f}{now:16,}{then:16,}{additional[name]:11,}{f'{lo:,}-{hi:,}':>14}")
print(f"\nTotal additional classrooms across the five districts: {sum(additional.values()):,}")

The last column converts the 95% prediction interval for 2029 into a classroom range. Treat the point estimates as planning figures, not commitments; the range shows the part of the uncertainty that the bootstrap can see.

## Extension A: Bootstrap prediction intervals
`Forecaster.bootstrap_interval` implements a **residual bootstrap** (2,000 resamples, seeded): (1) take the in-sample residuals and centre them to mean zero (the CAGR curve is pinned to the first and last points, so its raw residuals are biased, which I found by seeing off-centre intervals); (2) resample them with replacement, add them to the fitted curve to make a pseudo-series, **refit**, and forecast; (3) add one more resampled residual to every forecast year to represent future noise; (4) take the 2.5th and 97.5th percentiles. Refitting captures parameter uncertainty, the added residual captures noise.

In [ ]:
print(f"{'district':<9}{'2029 forecast':>14}{'95% interval':>20}{'half-width %':>14}")
for name in districts:
    iv = intervals[name]
    half = (iv["upper"][-1] - iv["lower"][-1]) / 2 / iv["point"][-1]
    print(f"{name:<9}{iv['point'][-1]:14,.0f}{f"{iv['lower'][-1]:,.0f} - {iv['upper'][-1]:,.0f}":>20}{half:14.1%}")

**Honest caveats.** The intervals are narrow (about ±1–5%) because they reflect only the scatter of the ten observations around a smooth curve. They do **not** include the chance that the growth rate itself changes (a slowdown like Jinja's, a policy change, a census revision), so they are *optimistic*. Residuals from an in-sample fit are also slightly too small, and ten points is a small basis for resampling. For comparison, the 2022–24 test errors of the winning CAGR model were 0.4–3.3% (MAPE) over just three years.

## Extension B: Is the Fibonacci-ratio model defensible?

In [ ]:
phi = (1 + 5 ** 0.5) / 2
fib = FibonacciRatioForecaster()
print("successive ratios F(k+1)/F(k):", fib.ratios(10).round(3).tolist(), f"-> golden ratio {phi:.3f}")
print(f"Applied every year, the converged ratio means growth of {phi - 1:.1%} per year.\n")

variants = {
    "literal (last x ratio_h)":                 FibonacciRatioForecaster(),
    "compounding (last x F(h+1))":              FibonacciRatioForecaster(chain=True),
    "compounding at converged ratio (1.618^h)": FibonacciRatioForecaster(start_index=20, chain=True),
}
print(f"{'district':<9}{'observed CAGR':>14}" + "".join(f"{v:>42}" for v in variants) + f"{'best model':>14}")
for name, d in districts.items():
    row = [ModelSelector([m], 2021).evaluate(d)[0].mape for m in variants.values()]
    best = ModelSelector(models[:2], 2021).best(ModelSelector(models[:2], 2021).evaluate(d))
    print(f"{name:<9}{d.cagr():14.2%}" + "".join(f"{r:41.1f}%" for r in row) + f"{best.mape:13.2f}%")
print("\n(MAPE on the 2022-24 test years; observed CAGR for comparison)")

**Assessment.**
- **What it assumes.** The Fibonacci ratios converge to 1.618, i.e. about **+61.8% per period, forever**. Observed district growth is **2–6.5% a year**, an order of magnitude lower. Under every reading the validation error is 36% or more (MAPE) against 0.4–3.3% for the CAGR model.
- **The early ratios oscillate** (1, 2, 1.5, 1.67, 1.6, …), so the literal model produces forecasts that jump up and down for a population that grows smoothly. The converged-ratio reading is smooth but absurdly steep.
- **It has nothing to calibrate.** 1.618 is a mathematical constant, not an estimate from the data; the model cannot adapt to a district growing at 2% or at 7%. It provides no uncertainty estimate either (there are no residuals to bootstrap, which is why `bootstrap_interval` raises `NotImplementedError` for it).
- **When would it be defensible?** Only if the quantity genuinely multiplies by about 1.618 each period: Fibonacci's idealised rabbit problem (unlimited food and space, no deaths, fixed generation time, each pair breeding from its second month), or a system whose next value is the sum of the previous two. A human population with births, deaths, migration and limited housing and services does not meet these conditions. If a Fibonacci-type pattern appeared in a district, a sensible approach would still be to *estimate* the growth factor from data, which is exactly what the CAGR model does.

## Findings & Limitations
**Findings.** On the illustrative data, Wakiso is the fastest-growing district (CAGR 6.47% a year) and Kampala and Gulu tie at 4.61%. After training on 2015–2021 and testing on 2022–2024, the CAGR (exponential) model beat the linear trend in all five districts, with test MAPE of 0.4–3.3% against 2.2–4.0% for the line, while the Fibonacci-ratio model was wrong by 36–47% and is not defensible for human populations. Applying the winners to 2025–2029 implies roughly 4,700 additional primary classrooms across the five districts (Wakiso about 2,090, Kampala about 1,540), subject to the stated assumptions. The forecast series has a different variance from the actual series mostly because it spans fewer points and a steeper slope, and because it contains no noise, so variance of a forecast should not be read as a risk measure. The bootstrap intervals (about ±1–5% in 2029) are therefore a lower bound on true uncertainty.

**Limitations.** Mbarara and Jinja are invented; all series are illustrative, so no real planning decision should rest on them. Validation uses only three test years, so differences such as Jinja's (RMSE 19.4 vs 20.5) are not conclusive, and Jinja's slowing growth means both models probably overshoot, so its 206 classrooms may be too many. CAGR assumes a constant growth rate and ignores migration, fertility change and carrying limits. The classroom estimate assumes today's stock exactly matches today's need and that the 18% school-age share and enrolment stay fixed.